# Communication Cost Analysis for Federated GNN Methods

This notebook provides an interactive interface for calculating and comparing communication costs across:
- **FedGCN** (1-hop expansion)
- **FedGAT Matrix** variant
- **FedGAT Vector** variant

For various datasets and partition schemes.

## Setup

In [ ]:
import os, sys, pathlib

# Add project root to path
project_root = pathlib.Path("/home/brian_bosho/FP/FP/federated-gnn")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("CWD:", os.getcwd())
print("Project root in sys.path:", str(project_root) in sys.path)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.communication_cost_analysis import (
    analyze_partition,
    run_communication_cost_analysis,
    print_summary_report,
    save_results,
    sci_notation
)
from omegaconf import OmegaConf

# Plotting setup
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load config
config_path = project_root / "conf" / "base.yaml"
config = OmegaConf.load(str(config_path))
print("Config loaded successfully")

## Quick Analysis: Single Configuration

In [ ]:
# Analyze a single partition scheme
result = analyze_partition(
    dataset_name="Cora",
    num_clients=10,
    beta=1.0,
    hop=1,
    device=device,
    config=config
)

# Display results
print("\n" + "="*80)
print("SINGLE CONFIGURATION ANALYSIS")
print("="*80)
print(f"Dataset: {result['dataset_name']}")
print(f"Clients: {result['num_clients']}, Beta: {result['beta']}, Hop: {result['hop']}")
print(f"\nGraph Statistics:")
print(f"  Total nodes: {result['total_nodes']}")
print(f"  Total edges: {result['total_edges']}")
print(f"  Feature dim: {result['feature_dim']}")
print(f"\nPartition Statistics:")
print(f"  Initial nodes (total): {result['total_initial_nodes']}")
print(f"  Expanded nodes (total): {result['total_expanded_nodes']}")
print(f"  New nodes added: {result['total_new_nodes']}")
print(f"  Expansion ratio: {result['expansion_ratio']:.2f}x")
print(f"\nCommunication Costs (scalars):")
print(f"  FedGCN:        {sci_notation(result['fedgcn_total'])}")
print(f"    Upload:      {sci_notation(result['fedgcn_upload'])}")
print(f"    Download:    {sci_notation(result['fedgcn_download'])}")
print(f"  FedGAT Matrix: {sci_notation(result['fedgat_matrix_total'])}")
print(f"  FedGAT Vector: {sci_notation(result['fedgat_vector_total'])}")
print(f"\nCost Ratios:")
print(f"  FedGAT Matrix / FedGCN: {result['fedgat_matrix_total'] / result['fedgcn_total']:.2f}x")
print(f"  FedGAT Vector / FedGCN: {result['fedgat_vector_total'] / result['fedgcn_total']:.2f}x")
print(f"  FedGAT Vector / Matrix: {result['fedgat_vector_total'] / result['fedgat_matrix_total']:.2f}x")

## Comprehensive Analysis: Multiple Configurations

In [ ]:
# Run comprehensive analysis
df_results = run_communication_cost_analysis(
    datasets=["Cora", "Citeseer", "Pubmed"],
    client_counts=[5, 10, 20],
    betas=[0.5, 1.0, 10.0],
    hops=[1],
    device="cpu",
    config_path=str(config_path)
)

In [ ]:
# Display summary report
print_summary_report(df_results)

In [ ]:
# View full results table
display_cols = [
    "dataset_name", "num_clients", "beta", "hop",
    "total_nodes", "feature_dim", "expansion_ratio",
    "fedgcn_total_exp", "fedgat_matrix_total_exp", "fedgat_vector_total_exp",
    "fedgat_matrix_vs_fedgcn", "fedgat_vector_vs_fedgcn"
]

df_results[display_cols]

## Visualizations

In [ ]:
# Plot 1: Communication costs by dataset and number of clients
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, dataset in enumerate(df_results["dataset_name"].unique()):
    df_dataset = df_results[(df_results["dataset_name"] == dataset) & (df_results["beta"] == 1.0)]
    
    ax = axes[idx]
    
    x = df_dataset["num_clients"]
    
    ax.plot(x, df_dataset["fedgcn_total"], marker='o', label="FedGCN", linewidth=2)
    ax.plot(x, df_dataset["fedgat_matrix_total"], marker='s', label="FedGAT Matrix", linewidth=2)
    ax.plot(x, df_dataset["fedgat_vector_total"], marker='^', label="FedGAT Vector", linewidth=2)
    
    ax.set_xlabel("Number of Clients", fontsize=12)
    ax.set_ylabel("Communication Cost (scalars)", fontsize=12)
    ax.set_title(f"{dataset} (beta=1.0)", fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig(project_root / "results" / "comm_cost_by_clients.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Cost ratios (FedGAT variants vs FedGCN)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter for beta=1.0
df_plot = df_results[df_results["beta"] == 1.0]

# Plot 2a: FedGAT Matrix / FedGCN
for dataset in df_plot["dataset_name"].unique():
    df_dataset = df_plot[df_plot["dataset_name"] == dataset]
    axes[0].plot(df_dataset["num_clients"], df_dataset["fedgat_matrix_vs_fedgcn"], 
                 marker='o', label=dataset, linewidth=2)

axes[0].set_xlabel("Number of Clients", fontsize=12)
axes[0].set_ylabel("Cost Ratio", fontsize=12)
axes[0].set_title("FedGAT Matrix / FedGCN", fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Equal cost')

# Plot 2b: FedGAT Vector / FedGCN
for dataset in df_plot["dataset_name"].unique():
    df_dataset = df_plot[df_plot["dataset_name"] == dataset]
    axes[1].plot(df_dataset["num_clients"], df_dataset["fedgat_vector_vs_fedgcn"], 
                 marker='s', label=dataset, linewidth=2)

axes[1].set_xlabel("Number of Clients", fontsize=12)
axes[1].set_ylabel("Cost Ratio", fontsize=12)
axes[1].set_title("FedGAT Vector / FedGCN", fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Equal cost')

plt.tight_layout()
plt.savefig(project_root / "results" / "comm_cost_ratios.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Effect of beta (Dirichlet concentration) on costs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fix num_clients=10
df_plot = df_results[df_results["num_clients"] == 10]

for idx, dataset in enumerate(df_plot["dataset_name"].unique()):
    df_dataset = df_plot[df_plot["dataset_name"] == dataset]
    
    ax = axes[idx]
    
    x = df_dataset["beta"]
    
    ax.plot(x, df_dataset["fedgcn_total"], marker='o', label="FedGCN", linewidth=2)
    ax.plot(x, df_dataset["fedgat_matrix_total"], marker='s', label="FedGAT Matrix", linewidth=2)
    ax.plot(x, df_dataset["fedgat_vector_total"], marker='^', label="FedGAT Vector", linewidth=2)
    
    ax.set_xlabel("Beta (Dirichlet concentration)", fontsize=12)
    ax.set_ylabel("Communication Cost (scalars)", fontsize=12)
    ax.set_title(f"{dataset} (10 clients)", fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig(project_root / "results" / "comm_cost_by_beta.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 4: Heatmap of cost ratios
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare data for heatmap (beta=1.0)
df_heat = df_results[df_results["beta"] == 1.0]

# Pivot for Matrix variant
pivot_matrix = df_heat.pivot_table(
    values='fedgat_matrix_vs_fedgcn',
    index='dataset_name',
    columns='num_clients'
)

# Pivot for Vector variant
pivot_vector = df_heat.pivot_table(
    values='fedgat_vector_vs_fedgcn',
    index='dataset_name',
    columns='num_clients'
)

sns.heatmap(pivot_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r', 
            ax=axes[0], cbar_kws={'label': 'Cost Ratio'})
axes[0].set_title('FedGAT Matrix / FedGCN', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Clients', fontsize=12)
axes[0].set_ylabel('Dataset', fontsize=12)

sns.heatmap(pivot_vector, annot=True, fmt='.2f', cmap='RdYlGn_r', 
            ax=axes[1], cbar_kws={'label': 'Cost Ratio'})
axes[1].set_title('FedGAT Vector / FedGCN', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Clients', fontsize=12)
axes[1].set_ylabel('Dataset', fontsize=12)

plt.tight_layout()
plt.savefig(project_root / "results" / "comm_cost_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
# Save to CSV
save_results(df_results, project_root / "results" / "communication_cost_analysis.csv")

## Custom Analysis

Use this section to run custom analyses with specific parameters.

In [ ]:
# Example: Analyze larger datasets
df_large = run_communication_cost_analysis(
    datasets=["Photo", "Computers"],  # Amazon datasets
    client_counts=[10, 20],
    betas=[1.0],
    hops=[1],
    device="cpu",
    config_path=str(config_path)
)

print_summary_report(df_large)

In [ ]:
# Example: Compare different hop values
df_hops = run_communication_cost_analysis(
    datasets=["Cora"],
    client_counts=[10],
    betas=[1.0],
    hops=[0, 1, 2],  # Compare 0-hop, 1-hop, 2-hop
    device="cpu",
    config_path=str(config_path)
)

print("\nEffect of k-hop expansion on communication cost:")
print(df_hops[['hop', 'expansion_ratio', 'fedgcn_total_exp', 
               'fedgat_matrix_total_exp', 'fedgat_vector_total_exp']])